In [64]:
import pandas as pd
import numpy as np

df_players = pd.read_csv('player_match_stats.csv')
df_matches = pd.read_csv('faceit_matches.csv')

In [65]:
match_extra = df_matches[[
    'match_id', 'score_t1', 'score_t2',
    't1_avg_elo', 't2_avg_elo',
    't1_avg_form', 't2_avg_form'
]].copy()

match_extra['total_rounds'] = match_extra['score_t1'] + match_extra['score_t2']

df = df_players.merge(match_extra, on='match_id', how='inner')

In [66]:
df.head()

,match_id,map,player_elo,player_kd,player_hs_pct,player_wr,player_matches,player_level,team_avg_elo,team_avg_kd,...,kills,deaths,assists,score_t1,score_t2,t1_avg_elo,t2_avg_elo,t1_avg_form,t2_avg_form,total_rounds
0,1-169c7c63-7622-4c71-91b3-a7f7f4979d9e,de_nuke,3222.0,1.04,42.0,52.0,6488.0,10.0,2852.8,1.086,...,29.0,18.0,3.0,16.0,14.0,2879.8,2857.0,0.8,0.52,30.0
1,1-169c7c63-7622-4c71-91b3-a7f7f4979d9e,de_nuke,2775.0,1.05,51.0,52.0,6006.0,10.0,2852.8,1.086,...,19.0,20.0,6.0,16.0,14.0,2879.8,2857.0,0.8,0.52,30.0
2,1-169c7c63-7622-4c71-91b3-a7f7f4979d9e,de_nuke,2869.0,1.10,49.0,53.0,2057.0,10.0,2852.8,1.086,...,22.0,24.0,3.0,16.0,14.0,2879.8,2857.0,0.8,0.52,30.0
3,1-169c7c63-7622-4c71-91b3-a7f7f4979d9e,de_nuke,2689.0,1.14,40.0,52.0,2389.0,10.0,2852.8,1.086,...,21.0,24.0,5.0,16.0,14.0,2879.8,2857.0,0.8,0.52,30.0
4,1-169c7c63-7622-4c71-91b3-a7f7f4979d9e,de_nuke,2709.0,1.10,50.0,51.0,3275.0,10.0,2852.8,1.086,...,19.0,24.0,6.0,16.0,14.0,2879.8,2857.0,0.8,0.52,30.0


# data quality checks

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   match_id        100000 non-null  object 
 1   map             100000 non-null  object 
 2   player_elo      100000 non-null  float64
 3   player_kd       100000 non-null  float64
 4   player_hs_pct   100000 non-null  float64
 5   player_wr       100000 non-null  float64
 6   player_matches  100000 non-null  float64
 7   player_level    100000 non-null  float64
 8   team_avg_elo    100000 non-null  float64
 9   team_avg_kd     100000 non-null  float64
 10  opp_avg_elo     100000 non-null  float64
 11  opp_avg_kd      100000 non-null  float64
 12  opp_avg_wr      100000 non-null  float64
 13  elo_gap         100000 non-null  float64
 14  kills           100000 non-null  float64
 15  deaths          100000 non-null  float64
 16  assists         100000 non-null  float64
 17  score_t1   

# Determining which team player belongs to

In [68]:
is_team1 = (df['team_avg_elo'] - df['t1_avg_elo']).abs() < 1.0

df['team_form'] = np.where(is_team1, df['t1_avg_form'], df['t2_avg_form'])
df['opp_form'] = np.where(is_team1, df['t2_avg_form'], df['t1_avg_form'])
df['form_advantage'] = df['team_form'] - df['opp_form']

# cleaning

removing matches with zero kills and zero deaths

In [69]:
print(len(df))
df = df[(df['kills'] > 0) | (df['deaths'] > 0)]
df = df[df['player_kd'] > 0]
df = df[(df['total_rounds'] >= 13) & (df['total_rounds'] <= 40)]
print(len(df))

100000
98842


removing extreme outliers (very unusual scores)

In [70]:
df = df[df['kills'] <= 50]
df = df[df['deaths'] <= 50]

# Creating target variable

In [71]:
df['match_kd'] = df['kills'] / df['deaths']

df['overperform'] = (df['match_kd'] > df['player_kd'] * 0.85).astype(int)

print("Target distribution:")
print(df['overperform'].value_counts())
print(f"\nBalance: {df['overperform'].mean():.3f} overperform rate")

Target distribution:
overperform
1    53804
0    45037
Name: count, dtype: int64

Balance: 0.544 overperform rate


# Feature engineering

In [72]:
df['elo_advantage_ratio'] = df['player_elo'] / df['opp_avg_elo'].clip(lower=1)

df['carry_factor'] = df['player_elo'] / df['team_avg_elo'].clip(lower=1)

df['elo_gap_abs'] = df['elo_gap'].abs()

df['team_kd_support'] = df['team_avg_kd'] - df['player_kd']

df['opp_kd_challenge'] = df['opp_avg_kd'] / df['player_kd'].clip(lower=0.1)

df['is_experienced'] = (df['player_matches'] > 500).astype(int)

df['wr_advantage'] = df['player_wr'] - df['opp_avg_wr']

# Training

In [73]:
from sklearn.model_selection import train_test_split

feature_cols = [
    'player_elo', 'player_kd', 'player_hs_pct', 'player_wr',
    'player_matches', 'player_level', 'team_avg_elo', 'team_avg_kd',
    'opp_avg_elo', 'opp_avg_kd', 'opp_avg_wr', 'elo_gap',
    'elo_advantage_ratio', 'carry_factor', 'elo_gap_abs',
    'team_kd_support', 'opp_kd_challenge', 'is_experienced', 'wr_advantage',
    'team_form', 'opp_form', 'form_advantage'
]

cat_cols = ['map']
num_cols = [c for c in feature_cols if c not in cat_cols]

X = df[num_cols + cat_cols]
y = df['overperform']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Preprocessor

In [74]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

# Baseline

In [75]:
majority_class = y_train.mode()[0]
baseline_acc = max(y_test.mean(), 1 - y_test.mean())
print(f"Majority class: {majority_class}")
print(f"Baseline accuracy (always predict majority): {baseline_acc:.3f}")

Majority class: 1
Baseline accuracy (always predict majority): 0.544


# Logistic regression

In [82]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_proba_lr = lr_pipe.predict_proba(X_test)[:, 1]
cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring='accuracy')

print('Logistic Regression:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_lr):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_lr):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_lr):.3f}')
print(f'CV Mean:   {cv_lr.mean():.3f} (+/- {cv_lr.std():.3f})')

Logistic Regression:
Accuracy:  0.623
F1 Score:  0.677
AUC:       0.661
CV Mean:   0.616 (+/- 0.002)


# RandomForest

In [83]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300, max_depth=12,
        min_samples_leaf=20, random_state=42
    ))
])

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
y_proba_rf = rf_pipe.predict_proba(X_test)[:, 1]
cv_rf = cross_val_score(rf_pipe, X_train, y_train, cv=5, scoring='accuracy')

print('Random Forest:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_rf):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_rf):.3f}')
print(f'CV Mean:   {cv_rf.mean():.3f} (+/- {cv_rf.std():.3f})')

Random Forest:
Accuracy:  0.620
F1 Score:  0.675
AUC:       0.655
CV Mean:   0.613 (+/- 0.003)


# GradientBoosting

In [84]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        min_samples_leaf=20, subsample=0.8, random_state=42
    ))
])

gb_pipe.fit(X_train, y_train)
y_pred_gb = gb_pipe.predict(X_test)
y_proba_gb = gb_pipe.predict_proba(X_test)[:, 1]
cv_gb = cross_val_score(gb_pipe, X_train, y_train, cv=5, scoring='accuracy')

print('Gradient Boosting:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_gb):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_gb):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_gb):.3f}')
print(f'CV Mean:   {cv_gb.mean():.3f} (+/- {cv_gb.std():.3f})')

Gradient Boosting:
Accuracy:  0.621
F1 Score:  0.673
AUC:       0.659
CV Mean:   0.616 (+/- 0.003)


# Neural Network

In [85]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

input_dim = X_train_processed.shape[1]

nn_model = Sequential()
nn_model.add(Dense(128, input_shape=(input_dim,)))
nn_model.add(BatchNormalization())
nn_model.add(Activation('relu'))
nn_model.add(Dropout(0.3))
nn_model.add(Dense(64))
nn_model.add(BatchNormalization())
nn_model.add(Activation('relu'))
nn_model.add(Dropout(0.2))
nn_model.add(Dense(32))
nn_model.add(Activation('relu'))
nn_model.add(Dropout(0.1))
nn_model.add(Dense(1))
nn_model.add(Activation('sigmoid'))

nn_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)

nn_model.fit(
    X_train_processed, y_train,
    epochs=300,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5804 - loss: 0.6745 - val_accuracy: 0.5996 - val_loss: 0.6630 - learning_rate: 0.0010
Epoch 2/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6004 - loss: 0.6627 - val_accuracy: 0.6115 - val_loss: 0.6568 - learning_rate: 0.0010
Epoch 3/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6081 - loss: 0.6586 - val_accuracy: 0.6159 - val_loss: 0.6558 - learning_rate: 0.0010
Epoch 4/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6095 - loss: 0.6569 - val_accuracy: 0.6137 - val_loss: 0.6551 - learning_rate: 0.0010
Epoch 5/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6113 - loss: 0.6555 - val_accuracy: 0.6122 - val_loss: 0.6562 - learning_rate: 0.0010
Epoch 6/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6147 - loss: 0.6539 - val_accuracy: 0.6163 - val_loss: 0.6536 - learning_rate: 0.0010
Epoch 7/300
495/495 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6141 - loss: 0.

In [86]:
y_proba_nn = nn_model.predict(X_test_processed).flatten()
y_pred_nn = (y_proba_nn >= 0.5).astype(int)

print('Neural Network:')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_nn):.3f}')
print(f'F1 Score:  {f1_score(y_test, y_pred_nn):.3f}')
print(f'AUC:       {roc_auc_score(y_test, y_proba_nn):.3f}')

618/618 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Neural Network:
Accuracy:  0.622
F1 Score:  0.673
AUC:       0.661


In [87]:
print(f"{'Model':<25} {'Accuracy':<12} {'F1':<12} {'AUC':<12} {'vs Baseline':<12}")
print("=" * 73)
print(f"{'BASELINE (majority)':<25} {baseline_acc:<12.3f} {'---':<12} {'---':<12} {'---':<12}")

models = {
    'Logistic Regression': (y_pred_lr, y_proba_lr),
    'Random Forest': (y_pred_rf, y_proba_rf),
    'Gradient Boosting': (y_pred_gb, y_proba_gb),
    'Neural Network': (y_pred_nn, y_proba_nn),
}

for name, (pred, proba) in models.items():
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)
    improvement = acc - baseline_acc
    print(f"{name:<25} {acc:<12.3f} {f1:<12.3f} {auc:<12.3f} {'+' if improvement > 0 else ''}{improvement:<12.3f}")

Model                     Accuracy     F1           AUC          vs Baseline 
BASELINE (majority)       0.544        ---          ---          ---         
Logistic Regression       0.623        0.677        0.661        +0.079       
Random Forest             0.620        0.675        0.655        +0.076       
Gradient Boosting         0.621        0.673        0.659        +0.076       
Neural Network            0.622        0.673        0.661        +0.077       


In [88]:
feature_names = preprocessor.get_feature_names_out()

importances = gb_pipe.named_steps['classifier'].feature_importances_

print("Feature importance:")
print("=" * 60)
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = '█' * int(imp * 200)
    is_new = '  ← NEW' if any(f in name for f in ['form', 'form_advantage']) else ''
    print(f"  {name:<35} {imp:.4f} {bar}{is_new}")

Feature importance for overperformance prediction:
  num__elo_advantage_ratio            0.2050 █████████████████████████████████████████
  num__elo_gap                        0.1567 ███████████████████████████████
  num__carry_factor                   0.1394 ███████████████████████████
  num__player_kd                      0.0633 ████████████
  num__opp_avg_kd                     0.0531 ██████████
  num__elo_gap_abs                    0.0470 █████████
  num__team_avg_kd                    0.0413 ████████
  num__player_matches                 0.0392 ███████
  num__team_avg_elo                   0.0356 ███████
  num__opp_avg_wr                     0.0316 ██████
  num__opp_avg_elo                    0.0315 ██████
  num__opp_kd_challenge               0.0275 █████
  num__team_kd_support                0.0257 █████
  num__player_elo                     0.0199 ███
  num__wr_advantage                   0.0176 ███
  num__player_hs_pct                  0.0160 ███
  num__form_advantage         

# Saving the model

In [89]:
import pickle

with open('overperform_classifier.pkl', 'wb') as f:
    pickle.dump(lr_pipe, f)

print('Model saved: goverperform_classifier.pkl')
print(f'Expected features: {num_cols + cat_cols}')

Model saved: goverperform_classifier.pkl
Expected features: ['player_elo', 'player_kd', 'player_hs_pct', 'player_wr', 'player_matches', 'player_level', 'team_avg_elo', 'team_avg_kd', 'opp_avg_elo', 'opp_avg_kd', 'opp_avg_wr', 'elo_gap', 'elo_advantage_ratio', 'carry_factor', 'elo_gap_abs', 'team_kd_support', 'opp_kd_challenge', 'is_experienced', 'wr_advantage', 'team_form', 'opp_form', 'form_advantage', 'map']
